# Search Engines — First Contact

**Mental model:** a relational database finds rows where a column EQUALS a value. A search engine finds documents where text CONTAINS or is RELEVANT to your query — and ranks them by how relevant they are. Elasticsearch builds an inverted index: for every word it knows which documents contain it. "Find all alerts mentioning CPU" is one query. Full-text relevance scoring, fuzzy matching, aggregations, and sub-second response on millions of documents are what Elasticsearch is built for.

## What makes search engines different

- **Inverted index** — maps every token to the documents containing it. The opposite of a row-based index.
- **Relevance scoring** — results ranked by TF-IDF or BM25. Not just "does it match" but "how well does it match".
- **Fuzzy matching** — finds "processer" even if you typed "processor". Edit distance tolerance built in.
- **Aggregations** — like GROUP BY but on free-text fields, nested objects, and across millions of documents fast.
- **When to use** — full-text search, log analytics, auto-complete, faceted search, any query where relevance ranking matters more than exact matching.

In [ ]:
from pathlib import Path
import sys
for _candidate in [Path('_setup'), Path('Basics/Databases/_setup')]:
    if _candidate.exists():
        sys.path.insert(0, str(_candidate.resolve()))
        break

from db_connections import get_elasticsearch_client
import pandas as pd
import json

es = get_elasticsearch_client()
info = es.info()
print(f"Connected to Elasticsearch")
print(f"Version: {info['version']['number']}")
print(f"Cluster: {info['cluster_name']}")

# Check our index exists and has documents
stats = es.indices.stats(index='telemetry_alerts')
doc_count = stats['indices']['telemetry_alerts']['total']['docs']['count']
print(f"Index: telemetry_alerts — {doc_count:,} documents")

In [ ]:
# Mapping = Elasticsearch's schema
# Shows how each field is indexed (text, keyword, date, etc.)
mapping = es.indices.get_mapping(index='telemetry_alerts')
props = mapping['telemetry_alerts']['mappings'].get('properties', {})
print("Index mapping (field → type):")
for field, config in sorted(props.items()):
    ftype = config.get('type', 'object')
    print(f"  {field:<20} {ftype}")

## 5 queries — Elasticsearch style

Note: Elasticsearch queries are JSON documents sent to the search API. The Query DSL is verbose but composable — every query type is a building block you can combine.

In [ ]:
# Find alerts where message contains "CPU" — relevance ranked
# SQL equivalent: WHERE message ILIKE '%cpu%' (but without ranking)
result = es.search(
    index='telemetry_alerts',
    body={
        "query": {
            "match": {
                "message": "CPU exceeded threshold"
            }
        },
        "size": 5,
        "_source": ["message", "severity", "category"]
    }
)
hits = result['hits']['hits']
print(f"Full-text search: 'CPU exceeded threshold'")
print(f"Total matches: {result['hits']['total']['value']:,}")
print(f"\nTop {len(hits)} results (ranked by relevance):")
for i, hit in enumerate(hits, 1):
    src = hit['_source']
    score = round(hit['_score'], 3)
    print(f"  {i}. [{score}] [{src.get('severity','?')}] {src.get('message','')[:80]}")

In [ ]:
# Find critical alerts mentioning "memory"
# bool query combines must (required) and filter (exact match, no scoring)
result = es.search(
    index='telemetry_alerts',
    body={
        "query": {
            "bool": {
                "must": {
                    "match": {"message": "memory"}
                },
                "filter": {
                    "term": {"severity": "critical"}
                }
            }
        },
        "size": 5,
        "_source": ["message", "severity", "category", "status"]
    }
)
hits = result['hits']['hits']
print(f"Critical alerts mentioning 'memory': {result['hits']['total']['value']:,} total")
for hit in hits:
    src = hit['_source']
    print(f"  [{src.get('severity')}] [{src.get('category')}] {src.get('message','')[:70]}")

In [ ]:
# Count alerts by severity — fast even on millions of docs
# SQL equivalent: SELECT severity, COUNT(*) FROM alerts GROUP BY severity
result = es.search(
    index='telemetry_alerts',
    body={
        "size": 0,
        "aggs": {
            "by_severity": {
                "terms": {"field": "severity", "size": 10}
            },
            "by_category": {
                "terms": {"field": "category", "size": 10}
            }
        }
    }
)
print("Alerts by severity:")
for bucket in result['aggregations']['by_severity']['buckets']:
    bar = '#' * (bucket['doc_count'] // 500)
    print(f"  {bucket['key']:<12} {bucket['doc_count']:>6,}  {bar}")

print("\nAlerts by category:")
for bucket in result['aggregations']['by_category']['buckets']:
    bar = '#' * (bucket['doc_count'] // 500)
    print(f"  {bucket['key']:<12} {bucket['doc_count']:>6,}  {bar}")

In [ ]:
# Find alerts even when the search term has a typo
# SQL LIKE cannot do this — ES fuzzy matching uses edit distance
result = es.search(
    index='telemetry_alerts',
    body={
        "query": {
            "fuzzy": {
                "message": {
                    "value": "processer",
                    "fuzziness": "AUTO"
                }
            }
        },
        "size": 5,
        "_source": ["message", "severity"]
    }
)
print(f"Fuzzy search for 'processer' (typo for 'processor'):")
print(f"Total matches: {result['hits']['total']['value']:,}")
for hit in result['hits']['hits']:
    src = hit['_source']
    print(f"  [{src.get('severity')}] {src.get('message','')[:80]}")

In [ ]:
# Search across message AND category fields
# AND filter to open alerts only in the last 30 days
result = es.search(
    index='telemetry_alerts',
    body={
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": "disk network",
                        "fields": ["message", "category"],
                        "type": "best_fields"
                    }
                },
                "filter": [
                    {"term": {"status": "open"}},
                    {"range": {
                        "created_at": {
                            "gte": "now-30d/d"
                        }
                    }}
                ]
            }
        },
        "size": 5,
        "_source": ["message", "severity", "category", "status"]
    }
)
print(f"Open alerts about 'disk network' in last 30 days: {result['hits']['total']['value']:,}")
for hit in result['hits']['hits']:
    src = hit['_source']
    score = round(hit['_score'], 3)
    print(f"  [{score}] [{src.get('category')}] {src.get('message','')[:70]}")

## SQL vs Elasticsearch Query DSL

| SQL | Elasticsearch |
|-----|---------------|
| `WHERE message ILIKE '%cpu%'` | `"match": {"message": "cpu"}` |
| `WHERE severity = 'critical'` | `"term": {"severity": "critical"}` |
| `WHERE message ILIKE '%cpu%' AND severity='critical'` | `"bool": {"must": match, "filter": term}` |
| `GROUP BY severity` | `"aggs": {"terms": {"field": "severity"}}` |
| `WHERE message ILIKE '%processer%'` | `"fuzzy": {"value": "processer", "fuzziness": "AUTO"}` |
| `WHERE created_at > NOW()-'30d'` | `"range": {"created_at": {"gte": "now-30d"}}` |
| ORDER BY relevance (no native support) | Default: ranked by BM25 relevance score |

## Key observations

- **Inverted index vs row scan** — SQL `ILIKE '%cpu%'` scans every row. ES `match` query hits the inverted index directly. At 25K documents the difference is small. At 25 million log lines the difference is everything.
- **text vs keyword fields** — `text` fields are analyzed (tokenized, lowercased, stemmed) for full-text search. `keyword` fields are stored as-is for exact match and aggregations. A field can have both (multi-field mapping).
- **Relevance score (_score)** — BM25 rewards term frequency and penalizes common words. The score tells you HOW well a document matches, not just WHETHER it matches.
- **bool query is the Swiss army knife** — `must` (scored match), `filter` (unscored exact), `should` (optional boost), `must_not` (exclusion). Combine any way you need.
- **Citi hook** — 25K alerts/day across 6K endpoints. An on-call engineer types "SSL certificate" at 3am. ES returns all SSL-related alerts ranked by relevance in under 10ms. SQL would need ILIKE on 25K rows with no ranking. Kibana at http://localhost:5601 visualizes this in real time.